# Transfer JSON files to NWB

This notebook converts Bonsai behavior-control JSON (or MAT) files into NWB format using the `bonsai_to_nwb` function from `TransferToNWB.py`.

**Environment note:** Run this notebook with the `Foraging` conda environment (or any environment where `pynwb`, `pandas`, `numpy`, `scipy` are installed with compatible binaries). The base Anaconda environment may have a NumPy 2.x binary-incompatibility issue.

## How to use
1. Set `INPUT_PATHS` to a list of JSON/MAT files and/or folders you want to convert.
2. Set `SAVE_FOLDER` to where you want the NWB files written.
3. Run all cells.

## 1. Imports and setup

In [6]:
import logging
import os
import sys
from glob import glob

# Make sure the package is importable when running from the repo
src_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from foraging_gui.TransferToNWB import bonsai_to_nwb

# Show progress / results from the converter
logger = logging.getLogger("foraging_gui.TransferToNWB")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

## 2. Configure inputs and output

`INPUT_PATHS` accepts individual files and/or folders. Folders are searched (recursively) for `.json` and `.mat` files.

In [ ]:
# Files and/or folders to convert
INPUT_PATHS = [
    r"I:\NWB\860083\860083_2026-09-17_14-46-45.json",
    # r"H:\test3\test.json",
    # r"H:\test3",              # a whole folder
    # r"H:\another\session.mat",
]

# Where to write the .nwb files
SAVE_FOLDER = r"I:\NWB\860083\"

# Search folders recursively for input files?
RECURSIVE = True


## 3. Collect the list of files to convert

In [8]:
def collect_files(paths, recursive=True):
    """Expand a list of files/folders into a sorted list of .json/.mat files."""
    files = []
    for p in paths:
        if os.path.isdir(p):
            pattern = "**/*" if recursive else "*"
            for ext in (".json", ".mat"):
                files.extend(
                    f
                    for f in glob(os.path.join(p, pattern + ext), recursive=recursive)
                )
        elif os.path.isfile(p):
            files.append(p)
        else:
            print(f"WARNING: path not found, skipping: {p}")
    return sorted(set(files))


files_to_convert = collect_files(INPUT_PATHS, recursive=RECURSIVE)
print(f"Found {len(files_to_convert)} file(s) to convert:")
for f in files_to_convert:
    print("  ", f)

Found 1 file(s) to convert:
   H:\test3\test.json


## 4. Run the conversion

`bonsai_to_nwb` returns a status string for each file:
- `success` — NWB written
- `empty_trials` — no trials found, nothing written
- `incomplete_json` — missing key fields, skipped
- `error` — an exception was raised (see message)

In [9]:
os.makedirs(SAVE_FOLDER, exist_ok=True)

results = []
for f in files_to_convert:
    try:
        status = bonsai_to_nwb(f, save_folder=SAVE_FOLDER)
    except Exception as e:
        status = f"error: {e}"
    results.append((f, status))
    print(f"{status:>16}  <-  {f}")

Successfully converted: H:\test3\results\test.nwb


         success  <-  H:\test3\test.json


## 5. Summary

In [10]:
from collections import Counter

counts = Counter(
    status.split(":")[0] for _, status in results
)
print("Conversion summary")
print("------------------")
for status, n in counts.items():
    print(f"{status}: {n}")

failed = [f for f, status in results if status.startswith("error")]
if failed:
    print("\nFiles that failed:")
    for f in failed:
        print("  ", f)

Conversion summary
------------------
success: 1
